In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   5. Single Source File: hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
   
   BUSINESS LOGIC: 
   - Denominator: N/A (Numerical Count).
   - Numerator: No of the unit's active third party vendors who are 1. single source 
     vendors AND 2. located in high risk jurisdictions.
   - Rule 1: There is a file filtering single source with engagement ID. Then use 
     engagement ID to find 3rd party location and country risk rating.
   - Rule 2: Col D, M, N: any of these three columns have high risk country, counted as 1.
   
   UPGRADES:
   - MASTER AU ANCHOR: Strictly outputs AUs from the filtered Master List.
   - TPRM MAPPING: Uses strict Databricks RLIKE with word boundaries (\b) to prevent 
     wildcard explosions when mapping string-to-string.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- 3. Data Quirk Corrected: 'Single Source' is stored in the EngagementNumber column.
    -- The actual E-Numbers (Engagement IDs) are stored in the ThirdPartyName column.
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 4. Extract TP data and flag if jurisdiction is completely missing
    SELECT 
        p.EngagementNumber,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    -- Bridge the real engagement ID to the main TP data table to keep only Single Source
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
),

High_Risk_Engagements AS (
    -- 5. Filter for Single Source engagements that touch a High Risk country
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Engagements_By_AU AS (
    -- 6. Map high-risk single source engagements using the TPRM unit mapping table
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT hr.EngagementNumber) AS High_Risk_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    
    -- FIXED JOIN: Uses Regex word boundaries and completely blocks blank mapping strings
    INNER JOIN High_Risk_Engagements hr
        ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
       AND (
           UPPER(hr.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
           OR UPPER(hr.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       )
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 7. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.Mapped_AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP04 Drill-Down
   
   PURPOSE: Shows EVERY single-source, high-risk third-party engagement, regardless 
   of whether the string mapped to an AU, or whether that AU exists in the Master List. 
   Useful for tracking unmapped high-risk sole source vendors.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        p.EngagementNumber,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
),

High_Risk_Engagements AS (
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using,
        tp.location_of_tp,
        tp.country_prod_serv_originates,
        tp.Td_Countries_Impacted,
        tp.Is_Missing_Jurisdiction,
        CASE 
            WHEN tp.Is_Missing_Jurisdiction = 1 THEN 'Missing All Jurisdictions'
            WHEN h1.CountryName IS NOT NULL THEN 'location_of_tp'
            WHEN h2.CountryName IS NOT NULL THEN 'country_prod_serv_originates'
            WHEN h3.CountryName IS NOT NULL THEN 'Td_Countries_Impacted'
        END AS Risk_Trigger
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
)

SELECT DISTINCT
    COALESCE(TRIM(CAST(map.Assessable_Unit_ID AS STRING)), 'UNMAPPED_ENGAGEMENT') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    hr.EngagementNumber,
    'Yes' AS Is_Single_Source,
    hr.owning_lob AS Raw_Col_K_owning_lob,
    hr.lob_using AS Raw_Col_L_lob_using,
    map.TPRM_Units AS Matched_Mapping_String,
    hr.Risk_Trigger
FROM High_Risk_Engagements hr

-- Join to mapping table using the safe RLIKE logic
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
   AND (
       UPPER(hr.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       OR UPPER(hr.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
   )
   
LEFT JOIN Master_AUs mast
    ON TRIM(CAST(map.Assessable_Unit_ID AS STRING)) = mast.BusinessID
ORDER BY BusinessID, hr.EngagementNumber;